# Vial ground-truth review, calibration, and validation

This notebook prepares human-review artifacts from all 26 processed videos. Candidate frames are suggestions only: nothing becomes a verified empty reference, physical calibration, or manual ground-truth label until a reviewer explicitly enters it.

In [ ]:
from pathlib import Path
import json, os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from vial_analysis import AnalysisConfig, analyze_batch
from vial_review import (
    estimate_mm_per_pixel, fit_volume_calibration, generate_review_artifacts,
    parse_verified_empty_flags, save_verified_empty_frames, validation_metrics,
)

ROOT = Path(os.getenv('VIAL_ROOT', '/home/hackathon5/vial_checks'))
RESULTS_DIR = ROOT / 'batch_results'
FRAMES_ROOT = ROOT / 'batch_work' / 'frames'
REVIEW_DIR = Path(os.getenv('VIAL_REVIEW_DIR', str(ROOT / 'review')))
VERIFIED_MAP_PATH = ROOT / 'verified_empty_frames.json'
COMBINED_PATH = RESULTS_DIR / 'combined_measurements.csv'

combined = pd.read_csv(COMBINED_PATH)
video_names = sorted(combined.video_name.unique())
artifact_paths = generate_review_artifacts(
    combined, FRAMES_ROOT, REVIEW_DIR, candidates_per_video=5, validation_per_video=6
)
candidates = pd.read_csv(artifact_paths['candidates'])
validation = pd.read_csv(artifact_paths['validation'])
calibration_points = pd.read_csv(artifact_paths['calibration'])
print(f'{len(video_names)} videos | {len(candidates)} empty-reference candidates | '      f'{len(validation)} validation rows')
print('Review directory:', REVIEW_DIR)

## 1. Confirm empty-vial reference frames

Choose a video below and inspect its first five candidate frames. In `empty_reference_candidates.csv`, set `verified_empty` to `True` only for frames where the vial is visibly and confidently empty. Confirm at least three frames for any video you want treated as absolute rather than relative-only. Leave uncertain frames blank or `False`.

In [ ]:
SELECTED_VIDEO = video_names[0]  # replace with another filename from video_names
selected_candidates = candidates[candidates.video_name == SELECTED_VIDEO]
display(selected_candidates)
fig, axes = plt.subplots(1, len(selected_candidates), figsize=(4 * len(selected_candidates), 4), squeeze=False)
for axis, (_, row) in zip(axes[0], selected_candidates.iterrows(), strict=True):
    image = cv2.cvtColor(cv2.imread(row.frame_path), cv2.COLOR_BGR2RGB)
    axis.imshow(image); axis.axis('off')
    axis.set_title(f"sample {row.frame_sampled} | {row.video_time_s:.2f}s\n{row.status}")
plt.tight_layout(); plt.show()

In [ ]:
# After editing and saving empty_reference_candidates.csv, rerun this cell.
candidates = pd.read_csv(artifact_paths['candidates'])
confirmed_map = parse_verified_empty_flags(candidates)
SAVE_VERIFIED_MAP = os.getenv('VIAL_SAVE_VERIFIED_MAP', '0') == '1'
if SAVE_VERIFIED_MAP:
    saved_map = save_verified_empty_frames(
        confirmed_map, VERIFIED_MAP_PATH, video_names, FRAMES_ROOT
    )
    print(f'Saved {len(saved_map)} verified video mappings to {VERIFIED_MAP_PATH}')
else:
    print(f'{len(confirmed_map)} videos currently have checked candidate rows.')
    print('Set VIAL_SAVE_VERIFIED_MAP=1 only after reviewing at least 3 frames per included video.')

## 2. Enter manual validation labels

Edit `manual_validation_sample.csv`. For each confidently measurable frame, enter `manual_fill_top_y` and `manual_fill_bottom_y`, then set `label_status` to `complete`. Use `rejected` for an unusable frame and leave other rows `unlabeled`. Optionally enter a known volume and notes. Metrics ignore every row not explicitly marked `complete`.

In [ ]:
validation = pd.read_csv(artifact_paths['validation'])
selected_validation = validation[validation.video_name == SELECTED_VIDEO]
display(selected_validation)
fig, axes = plt.subplots(1, len(selected_validation), figsize=(4 * len(selected_validation), 4), squeeze=False)
for axis, (_, row) in zip(axes[0], selected_validation.iterrows(), strict=True):
    image = cv2.cvtColor(cv2.imread(row.frame_path), cv2.COLOR_BGR2RGB)
    axis.imshow(image); axis.axis('off')
    axis.set_title(f"sample {row.frame_sampled} | {row.status}")
plt.tight_layout(); plt.show()
if validation.label_status.fillna('unlabeled').astype(str).str.lower().eq('complete').any():
    print('Completed labels found. Metrics are computed after the verified rerun below.')
else:
    print('No manual height labels entered yet; metrics remain unavailable.')

## 3. Fit physical calibration

Edit `calibration_points.csv` using real measurements. `known_height_mm / measured_height_px` determines scale. Rows that also contain `known_volume_ml` fit the height-to-volume curve. At least two distinct volume points are required for a linear fit; use more points across the operating range.

In [ ]:
calibration_points = pd.read_csv(artifact_paths['calibration'])
MM_PER_PIXEL = None
VOLUME_COEFFICIENTS = None
if calibration_points.known_height_mm.notna().any():
    MM_PER_PIXEL = estimate_mm_per_pixel(calibration_points)
    print('MM_PER_PIXEL =', MM_PER_PIXEL)
    if calibration_points.known_volume_ml.notna().any():
        VOLUME_COEFFICIENTS, fit_metrics = fit_volume_calibration(
            calibration_points, MM_PER_PIXEL, degree=1
        )
        print('VOLUME_COEFFICIENTS =', VOLUME_COEFFICIENTS)
        display(pd.Series(fit_metrics, name='value').to_frame())
else:
    print('No physical measurements entered; calibration remains unset.')

## 4. Run verified analysis

This cell is opt-in. It reads only the saved, validated JSON map and calibration values above. Unmapped videos remain relative-only. Set `VIAL_RUN_VERIFIED_BATCH=1` when you are ready; results are written separately to `verified_batch_results`.

In [ ]:
RUN_VERIFIED_BATCH = os.getenv('VIAL_RUN_VERIFIED_BATCH', '0') == '1'
if VERIFIED_MAP_PATH.exists():
    verified_map = {
        name: tuple(indices) for name, indices in json.loads(VERIFIED_MAP_PATH.read_text()).items()
    }
else:
    verified_map = {}
verified_config = AnalysisConfig(
    video_dir=ROOT / 'drive-download-20261031T172038Z-1-001',
    work_dir=ROOT / 'batch_work', results_dir=ROOT / 'verified_batch_results',
    stride=3, vial_box=(300, 60, 360, 420), reference_frame_count=15,
    minimum_reference_frames=3, verified_empty_frames=verified_map,
    mm_per_pixel=MM_PER_PIXEL, volume_coefficients=VOLUME_COEFFICIENTS, resume=True,
)
if RUN_VERIFIED_BATCH:
    verified_combined, verified_manifest = analyze_batch(verified_config)
    display(verified_manifest)
    validation = pd.read_csv(artifact_paths['validation'])
    keys = ['video_name', 'frame_sampled']
    prediction_columns = keys + ['fill_height_px', 'reference_basis']
    predictions = verified_combined[prediction_columns]
    if predictions.duplicated(keys).any():
        raise ValueError('Verified measurements contain duplicate video/frame keys')
    scored = validation.drop(columns=['fill_height_px', 'reference_basis'], errors='ignore').merge(
        predictions, on=keys, how='left', validate='one_to_one'
    )
    completed = scored.label_status.fillna('unlabeled').astype(str).str.lower().eq('complete')
    invalid_provenance = completed & scored.fill_height_px.notna() & scored.reference_basis.ne('verified_empty')
    if invalid_provenance.any():
        raise ValueError('Completed labels received predictions without verified-empty provenance')
    if completed.any():
        display(pd.Series(validation_metrics(scored), name='value').to_frame())
    else:
        print('Verified rerun finished; no labels are marked complete, so metrics remain unavailable.')
else:
    print(f'Verified batch disabled. Saved verified mappings: {len(verified_map)} videos.')

## Files to edit

- `review/empty_reference_candidates.csv`: mark confidently empty candidates.
- `review/manual_validation_sample.csv`: enter manual top/bottom rows and known volumes.
- `review/calibration_points.csv`: enter measured physical references.

The notebook intentionally stops here until those observations exist. Predictions alone are not ground truth.